# 07 Evokeds

This notebook creates condition-specific evoked responses from saved epochs.

Inputs:

- `desc-cleaned_epo.fif`
- `epochs.metadata`

Output:

- one file per condition in `evokeds/*_desc-<condition>_ave.fif`

The key idea is that conditions are defined in this notebook using metadata queries or old-style event ID lists. This keeps the Library generic and the project-specific experimental logic visible.


## Setup

In [ ]:
from __future__ import annotations

from pathlib import Path

import mne
import pandas as pd

from meeg_pipeline.config import load_config
from meeg_pipeline.evokeds import configured_conditions, make_evoked_path, write_evokeds_for_recordings
from meeg_pipeline.workflow import (
    evoked_condition_counts_to_dataframe,
    evoked_condition_results_to_dataframe,
    evoked_metadata_preview_to_dataframe,
    evoked_results_to_dataframe,
    evokeds_input_overview_to_dataframe,
    existing_output_policy_for_step,
    iter_recordings,
    safe_join,
    selected_recordings_to_dataframe,
    should_overwrite,
)

def find_project_root(start: Path | None = None) -> Path:
    """Find project root by searching upward for configs/local.yaml."""
    start = Path.cwd() if start is None else Path(start).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "local.yaml").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find project root by searching for configs/local.yaml "
        f"above {start}"
    )


PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "configs" / "local.yaml"
config = load_config(CONFIG_PATH)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CONFIG_PATH:", CONFIG_PATH)


## Interactive backend

In [ ]:
%matplotlib qt

mne.viz.set_browser_backend("qt")
mne.set_log_level("WARNING")

print("MNE browser backend:", mne.viz.get_browser_backend())


## Selection

In [ ]:
SUBJECTS = "all"
SESSIONS = "all"
TASKS = "all"
RUNS = "all"

# Single-file/manual inspection cells at the end of notebooks are disabled by default
# so batch runs over many participants do not stop for plots or ad-hoc file views.
RUN_SINGLE_FILE_INSPECTIONS = False
selected_recordings = list(
    iter_recordings(
        config,
        subjects=SUBJECTS,
        sessions=SESSIONS,
        tasks=TASKS,
        runs=RUNS,
    )
)

selected_recordings_to_dataframe(selected_recordings)


## Overwrite policy

Default:

    OVERWRITE_STEPS = []

Existing evoked files are skipped. To recompute evokeds:

    OVERWRITE_STEPS = ["evokeds"]


In [ ]:
OVERWRITE_STEPS = []

pd.DataFrame(
    [
        {
            "step": "evokeds",
            "overwrite": should_overwrite("evokeds", OVERWRITE_STEPS),
            "policy": existing_output_policy_for_step(
                "evokeds",
                OVERWRITE_STEPS,
            ),
        }
    ]
)


## Define conditions

Condition definitions are project-specific.

Recommended new workflow: use metadata queries.

Examples:

```python
"non_diatonic in [1, 2, 3, 4, 5]"
"note_index == 0"
"trial_type == 'non_diatonic'"
```

Old-style workflow is also supported by passing event-code lists:

```python
"some_condition": [1, 5, 9]
```


In [ ]:
CONDITIONS = configured_conditions(config)

if not CONDITIONS:
    raise ValueError(
        "No condition definitions found in configs/local.yaml under "
        "conditions.definitions. Define project-specific conditions there, "
        "or pass trigger/event-code conditions explicitly."
    )

pd.DataFrame(
    [
        {
            "condition": condition,
            "definition": definition,
        }
        for condition, definition in CONDITIONS.items()
    ]
)


## Input overview

In [ ]:
evokeds_input_overview = evokeds_input_overview_to_dataframe(
    config,
    selected_recordings,
    CONDITIONS,
)

evokeds_input_overview


## Preview metadata for one recording

In [ ]:
PREVIEW_INDEX = 0

PREVIEW = selected_recordings[PREVIEW_INDEX]

evoked_metadata_preview_to_dataframe(config, PREVIEW)


## Preview condition counts for one recording

In [ ]:
condition_count_preview = evoked_condition_counts_to_dataframe(
    config,
    PREVIEW,
    CONDITIONS,
)

condition_count_preview


## Write evokeds

In [ ]:
evokeds_policy = existing_output_policy_for_step(
    "evokeds",
    OVERWRITE_STEPS,
)

evoked_results = write_evokeds_for_recordings(
    config,
    selected_recordings,
    conditions=CONDITIONS,
    on_existing=evokeds_policy,
)

evoked_results_table = evoked_results_to_dataframe(
    selected_recordings,
    evoked_results,
)

evoked_results_table


## Condition-level write details

In [ ]:
condition_detail_table = evoked_condition_results_to_dataframe(
    selected_recordings,
    evoked_results,
)

condition_detail_table


## Inspect one condition-specific evoked file

> Disabled by default for batch runs. Set `RUN_SINGLE_FILE_INSPECTIONS = True` in the selection cell to run this section.



In [ ]:
if RUN_SINGLE_FILE_INSPECTIONS:
    INSPECT_SUBJECT = "0001"
    INSPECT_SESSION = None
    INSPECT_TASK = "example"
    INSPECT_RUN = None
    INSPECT_CONDITION = "example_condition"

    inspect_path = make_evoked_path(
        config,
        subject=INSPECT_SUBJECT,
        session=INSPECT_SESSION,
        task=INSPECT_TASK,
        run=INSPECT_RUN,
        condition=INSPECT_CONDITION,
    )

    if inspect_path.exists():
        evokeds = mne.read_evokeds(inspect_path, verbose="error")
        inspect_status = pd.DataFrame(
            [
                {
                    "status": "loaded",
                    "condition": INSPECT_CONDITION,
                    "n_evokeds": len(evokeds),
                    "comments": safe_join([evoked.comment for evoked in evokeds]),
                    "path": str(inspect_path),
                }
            ]
        )
    else:
        evokeds = []
        inspect_status = pd.DataFrame(
            [
                {
                    "status": "missing_input",
                    "condition": INSPECT_CONDITION,
                    "n_evokeds": 0,
                    "comments": "",
                    "path": str(inspect_path),
                }
            ]
        )

    inspect_status
else:
    print('Skipped single-file inspection cell 22 in 2_sensor_analysis/01_evokeds.ipynb. Set RUN_SINGLE_FILE_INSPECTIONS = True to run it.')


## Plot inspected evokeds

> Disabled by default for batch runs. Set `RUN_SINGLE_FILE_INSPECTIONS = True` in the selection cell to run this section.



In [ ]:
if RUN_SINGLE_FILE_INSPECTIONS:
    if "evokeds" not in globals() or not evokeds:
        print("No evokeds loaded. Run the previous inspection cell first.")
    else:
        for evoked in evokeds:
            evoked.plot(spatial_colors=True)
else:
    print('Skipped single-file inspection cell 24 in 2_sensor_analysis/01_evokeds.ipynb. Set RUN_SINGLE_FILE_INSPECTIONS = True to run it.')
